<a href="https://colab.research.google.com/github/mblci/Retina-Diseases-DeepLearning-Benchmark/blob/main/1_Augmented_and_Original_Dataset_3Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**tek satırla modeli değiştirebileceğin hazır eğitim kod şablonu**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================
# CITATION INFORMATION
# ============================================================
# If you find this code useful for your research, please cite:
#
# Muharrem BALCI, STATISTICAL RELIABILITY AND EXPLAINABILITY
# OF MODERN CONVNEXTV2 AND SWIN TRANSFORMER ARCHITECTURES IN THE
# CLASSIFICATION OF MULTIPLE RETINAL DISEASES BASED ON FUNDUS IMAGES,
# (Submitted for publication), 2026.
#
# GitHub: https://github.com/mblci/Retina-Diseases-DeepLearning-Benchmark
# ============================================================


# ============================================================
# STATISTICAL RELIABILITY AND EXPLAINABILITY
# OF MODERN CONVNEXTV2 AND SWIN TRANSFORMER ARCHITECTURES IN THE
# CLASSIFICATION OF MULTIPLE RETINAL DISEASES BASED ON FUNDUS IMAGES
# (Training 3 different Models With Augmented and Original Datasets)
# ============================================================
# Description: Data splitting, training (AMP), early stopping,
# history tracking, testing, confusion matrix, classification
# report, ROC/PR curves, misclassified examples, and param count.
# Dataset Source: [Eye Disease Image Dataset: https://data.mendeley.com/datasets/s9bfhswzjb/1]
# ============================================================

import os
import random
import shutil
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from timm import create_model
import torch.optim as optim
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

# -----------------------------
# 1-Device Configuration
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# -----------------------------
# 2-Data Paths and Splitting
# -----------------------------
# The local data directory or project folder is set here.
data_dir = "./dataset/augmented"
base_dir = "./dataset/split_data"
results_dir = "./results/dynamic_models_metrics"
os.makedirs(results_dir, exist_ok=True)

if not os.path.exists(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(base_dir, split), exist_ok=True)

    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        random.shuffle(images)

        n_total = len(images)
        n_train = int(0.85 * n_total)
        n_val = int(0.10 * n_total)
        n_test = n_total - n_train - n_val

        splits = {
            "train": images[:n_train],
            "val": images[n_train:n_train + n_val],
            "test": images[n_train + n_val:]
        }

        for split, imgs in splits.items():
            split_dir = os.path.join(base_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)
            for img in imgs:
                shutil.copy(os.path.join(cls_path, img),
                            os.path.join(split_dir, img))
    print("Data split completed (85/10/5 ratio).")
else:
    print("Split data already exists, skipping partitioning.")

# -----------------------------
# 3-Transforms & Safe Loader
# -----------------------------
def pil_loader(path):
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')
    except Exception as e:
        print(f"Error loading image: {path} ({e})")
        return Image.new("RGB", (224, 224), (0, 0, 0))

IMG_SIZE = 224
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_data = datasets.ImageFolder(root=os.path.join(base_dir, "train"), transform=train_transform, loader=pil_loader)
val_data = datasets.ImageFolder(root=os.path.join(base_dir, "val"), transform=val_test_transform, loader=pil_loader)
test_data = datasets.ImageFolder(root=os.path.join(base_dir, "test"), transform=val_test_transform, loader=pil_loader)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(train_data.classes)
print(f"Class count: {num_classes}, Class names: {train_data.classes}")

# -----------------------------
# 4- Model Selection
# -----------------------------
model_list = {
    "EfficientNetV2_S": "tf_efficientnetv2_s.in21k_ft_in1k",
    "Swin_Tiny": "swin_tiny_patch4_window7_224",
    "ConvNeXtV2_Base": "convnextv2_base",
}

model_name = "ConvNeXtV2_Base"
model_timm = model_list[model_name]
model = create_model(model_timm, pretrained=True, num_classes=num_classes)
model.to(device)
print(f"Using model: {model_name}")

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
with open(os.path.join(results_dir, f"model_info_{model_name.replace('/', '_')}.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Trainable parameters: {total_params:,}\n")

# -----------------------------
# 5- Loss, Optimizer, Scheduler
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# -----------------------------
# 6- Training Loop (AMP) + EarlyStopping + History
# -----------------------------
epochs = 15
best_val_loss = float('inf')
patience = 10
counter = 0
scaler = GradScaler()

history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    print(f"\nEpoch [{epoch+1}/{epochs}]")
    model.train()
    train_loss = 0.0
    running_samples = 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        running_samples += images.size(0)

    train_loss = train_loss / running_samples
    scheduler.step()

    # Validation
    model.eval()
    val_loss = 0.0
    val_samples = 0
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_samples += images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss / val_samples
    val_acc = correct / total
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # Early stopping + best model save
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), os.path.join(results_dir, f"best_{model_name.replace('/', '_')}.pth"))
        print(" Best model saved!")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered (No improvement for {patience} epochs).")
            break

# Save training history
pd.DataFrame(history).to_csv(os.path.join(results_dir, f"training_history_{model_name.replace('/', '_')}.csv"), index=False)

# Plot Learning Curves
plt.figure(figsize=(10,5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Loss Curve - {model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"loss_curve_{model_name.replace('/', '_')}.png"))
plt.show()

plt.figure(figsize=(10,5))
plt.plot(history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"Accuracy Curve - {model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"accuracy_curve_{model_name.replace('/', '_')}.png"))
plt.show()

# -----------------------------
# 7- Test Evaluation
# -----------------------------
model.eval()
y_true, y_pred = [], []
y_prob = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# Performance Metrics
metrics = {
    "Model": model_name,
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision (macro)": precision_score(y_true, y_pred, average='macro', zero_division=0),
    "Recall (macro)": recall_score(y_true, y_pred, average='macro', zero_division=0),
    "F1-Score (macro)": f1_score(y_true, y_pred, average='macro', zero_division=0)
}

print("\n Test Metrics:\n", metrics)
with open(os.path.join(results_dir, f"test_metrics_{model_name.replace('/', '_')}.txt"), "w") as f:
    f.write("=== TEST PERFORMANCE METRICS ===\n")
    for k, v in metrics.items():
        if isinstance(v, float) or isinstance(v, int):
            f.write(f"{k}: {v:.4f}\n")
        else:
            f.write(f"{k}: {v}\n")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=train_data.classes,
            yticklabels=train_data.classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {model_name}")
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"confusion_matrix_{model_name.replace('/', '_')}.png"))
plt.show()

# Classification Report
report = classification_report(y_true, y_pred, target_names=train_data.classes, digits=4, zero_division=0)
print("\nClassification Report:\n", report)
with open(os.path.join(results_dir, f"classification_report_{model_name.replace('/', '_')}.txt"), "w") as f:
    f.write(report)

# ROC Curve & AUC
y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
if y_true_bin.shape[1] != num_classes:
    y_true_bin = np.eye(num_classes)[y_true]

plt.figure(figsize=(8,6))
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{train_data.classes[i]} (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - {model_name}")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"roc_curve_{model_name.replace('/', '_')}.png"))
plt.show()

# Macro/Micro AUC
try:
    aucs = []
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        aucs.append(auc(fpr, tpr))
    macro_auc = np.mean(aucs)
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    micro_auc = auc(fpr_micro, tpr_micro)
    with open(os.path.join(results_dir, f"auc_summary_{model_name.replace('/', '_')}.txt"), "w") as f:
        f.write(f"Macro AUC: {macro_auc:.4f}\n")
        f.write(f"Micro AUC: {micro_auc:.4f}\n")
    print(f"Macro AUC: {macro_auc:.4f} | Micro AUC: {micro_auc:.4f}")
except Exception as e:
    print("Error calculating ROC AUC:", e)

# Precision-Recall Curve
plt.figure(figsize=(8,6))
for i in range(num_classes):
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_prob[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_prob[:, i])
    plt.plot(recall, precision, label=f"{train_data.classes[i]} (AP = {ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve - {model_name}")
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"pr_curve_{model_name.replace('/', '_')}.png"))
plt.show()

# -----------------------------
# 8- Save Misclassified Samples
# -----------------------------
misclf_dir = os.path.join(results_dir, f"misclassified_{model_name.replace('/', '_')}")
os.makedirs(misclf_dir, exist_ok=True)

def unnormalize_tensor(tensor):
    t = tensor.clone().cpu()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    t = t.permute(1,2,0).numpy()
    t = np.clip(t * 255.0, 0, 255).astype(np.uint8)
    return t

idx = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Saving Misclassified"):
        images_cpu = images
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        for i in range(images.size(0)):
            true = int(labels[i].item())
            pred = int(preds[i].cpu().item())
            if true != pred:
                img_arr = unnormalize_tensor(images_cpu[i])
                conf = float(probs[i, pred].cpu().item())
                fname = f"idx_{idx}_true_{train_data.classes[true]}_pred_{train_data.classes[pred]}_conf_{conf:.3f}.png"
                Image.fromarray(img_arr).save(os.path.join(misclf_dir, fname))
                idx += 1

print(f"Total misclassified images saved: {idx}")

# -----------------------------
# 9- Final Summary Export
# -----------------------------
report_dict = classification_report(y_true, y_pred, target_names=train_data.classes, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv(os.path.join(results_dir, f"classification_report_table_{model_name.replace('/', '_')}.csv"))

summary = {
    "model": model_name,
    "trainable_params": total_params,
    "accuracy": metrics["Accuracy"],
    "precision_macro": metrics["Precision (macro)"],
    "recall_macro": metrics["Recall (macro)"],
    "f1_macro": metrics["F1-Score (macro)"],
}
pd.Series(summary).to_csv(os.path.join(results_dir, f"summary_{model_name.replace('/', '_')}.csv"))

print(f"\n All outputs saved to: {results_dir}")
print(f" Execution completed for: {model_name}")